In [ ]:
import requests as req
from bs4 import BeautifulSoup as bs
from urllib.parse import urljoin
import time
import csv

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'id-ID,id;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}
session = req.Session()
session.headers.update(headers)
processed_urls = set()  # Track processed URLs to avoid duplicates

def print_header(title):
    print(f"\n{'=' * 50}")
    print(f"{title.upper():^50}")
    print(f"{'=' * 50}")

def print_status(message, status="INFO"):
    print(f"[{status}] {message}")

def analyze_structure(url):
    try:
        response = session.get(url)
        response.raise_for_status()
        soup = bs(response.text, 'lxml')

        print_header("HTML Structure Analysis")
        print(f"Page Title: {soup.title.text if soup.title else 'Not available'}")

        # Debug: Check for different possible containers
        container_checks = [
            ('div.articleList.-list', 'articleList -list'),
            ('div[class*="articleList"]', 'articleList variants'),
            ('div.latest__item', 'latest__item'),
            ('div.article__list--item', 'article__list--item'),
            ('div[class*="latest"]', 'latest variants'),
            ('div[class*="article"]', 'article variants'),
            ('article', 'article tags'),
            ('div[class*="item"]', 'item variants'),
            ('div[class*="list"]', 'list variants')
        ]
        
        print("\nContainer Detection:")
        for selector, description in container_checks:
            containers = soup.select(selector)
            print(f"- {description}: {len(containers)} found")
        
        # Find the best container type
        main_containers = []
        best_selector = None
        
        for selector, description in container_checks:
            containers = soup.select(selector)
            if containers and len(containers) > len(main_containers):
                main_containers = containers
                best_selector = selector
        
        print(f"\nBest Container: {best_selector}")
        print(f"Main Article Containers Found: {len(main_containers)}")

        if main_containers:
            sample = main_containers[0]
            print("\nSample Container Analysis:")
            print(f"- CSS Classes: {', '.join(sample.get('class', ['None']))}")

            # Find the main link in each article
            main_link = (
                sample.select_one('h3 a') or 
                sample.select_one('h2 a') or 
                sample.select_one('a[href*="/read/"]') or
                sample.select_one('a')
            )
            if main_link:
                print(f"- Article URL: {main_link.get('href', 'Not available')}")
                print(f"- Title Preview: {main_link.get_text(strip=True)[:50]}...")
            else:
                print("- No article links found in sample")

        return soup

    except Exception as e:
        print_status(f"Structure analysis failed: {e}", "ERROR")
        return None

def find_unique_articles(soup):
    articles = []

    # Multiple selectors for Kompas.com article containers (in priority order)
    container_selectors = [
        'div.articleList.-list',  # Most specific selector found in logs
        'div.latest__item',
        'div.article__list--item', 
        'div[class*="articleList"]',
        'div[class*="article"]',
        'div[class*="latest"]',
        'article',
        'div[class*="item"]',
        'div[class*="list"]'
    ]
    
    article_containers = []
    selector_used = None
    
    # Try each selector and take the one with most results
    best_containers = []
    best_selector = None
    
    for selector in container_selectors:
        containers = soup.select(selector)
        if containers and len(containers) > len(best_containers):
            best_containers = containers
            best_selector = selector
    
    if best_containers:
        article_containers = best_containers
        print_status(f"Using selector: {best_selector}")
    else:
        print_status("No article containers found with standard selectors")
        return articles
    
    print_status(f"Processing {len(article_containers)} article containers")

    for container in article_containers:
        try:
            # Find the main article link and title for Kompas (more aggressive search)
            title_link = None
            
            # Priority search for links
            link_selectors = [
                'h3 a',
                'h2 a', 
                'h4 a',
                'h1 a',
                'a[href*="/read/"]',
                'a[href*="kompas.com"]',
                'a[href*="/20"]',  # Year-based URLs
                'a[title]',  # Links with titles
                'a'
            ]
            
            for link_sel in link_selectors:
                title_link = container.select_one(link_sel)
                if title_link and title_link.get('href'):
                    break

            if not title_link or not title_link.get('href'):
                # Last resort: look for any text that might be a title
                potential_titles = container.find_all(text=True)
                if potential_titles:
                    for text in potential_titles:
                        text = text.strip()
                        if len(text) > 20 and not text.lower().startswith(('news', 'baca', 'lihat')):
                            # Try to find a nearby link
                            parent = container
                            for _ in range(3):  # Search up to 3 levels
                                nearby_link = parent.find('a', href=True)
                                if nearby_link:
                                    title_link = nearby_link
                                    break
                                parent = parent.parent if parent.parent else parent
                            break
                
                if not title_link:
                    continue

            # Get URL and normalize it
            url = title_link.get('href')
            if url.startswith('/'):
                url = urljoin('https://www.kompas.com', url)
            elif not url.startswith('http'):
                continue

            # Skip non-Kompas URLs or unwanted sections
            if 'kompas.com' not in url:
                continue
            
            # Skip if already processed (avoid duplicates)
            if url in processed_urls:
                continue

            # Get title (more flexible approach)
            title = title_link.get_text(strip=True)
            
            # If title is too short, try to get it from nearby elements
            if not title or len(title) < 10:
                # Look for title in parent elements
                for parent in [container, title_link.parent, title_link.parent.parent if title_link.parent else None]:
                    if not parent:
                        continue
                    text_content = parent.get_text(strip=True)
                    if len(text_content) > 10 and len(text_content) < 200:
                        title = text_content
                        break
                
                if not title or len(title) < 10:
                    continue

            # Get date from container (more flexible)
            date_selectors = [
                'div.article__date',
                'span.article__date', 
                'div[class*="date"]',
                'span[class*="date"]',
                'time',
                '.latest__date',
                '[datetime]',
                'small'  # Sometimes dates are in small tags
            ]
            
            date_elem = None
            for date_sel in date_selectors:
                date_elem = container.select_one(date_sel)
                if date_elem:
                    break
                    
            date = date_elem.get_text(strip=True) if date_elem else 'Not available'

            # Clean date for Kompas format
            date = date.replace('WIB', '').replace('WITA', '').replace('WIT', '').strip()
            if ',' in date:
                date = date.split(',')[-1].strip()

            # Clean title
            title = title.replace('\n', ' ').replace('\t', ' ')
            title = ' '.join(title.split())  # Remove extra spaces
            title = title[:200]  # Limit title length

            # Add to processed URLs
            processed_urls.add(url)

            articles.append({
                'title': title,
                'url': url,
                'date': date,
                'container': container
            })

        except Exception as e:
            print_status(f"Error processing container: {e}", "WARNING")
            continue

    return articles

def extract_article_content(url):
    try:
        time.sleep(1)  # Rate limiting
        response = session.get(url)
        response.raise_for_status()

        soup = bs(response.text, 'lxml')

        # Content selectors specific to Kompas.com (in priority order)
        content_selectors = [
            'div.read__content',
            'div[class*="read__content"]',
            'div.artikel__content',
            'div[class*="content"]',
            'div[class*="article-content"]',
            'div[class*="read-content"]',
            'div.content',
            'div#content'
        ]

        content = ""
        content_found = False

        for selector in content_selectors:
            elements = soup.select(selector)
            if elements:
                for elem in elements:
                    paragraphs = elem.find_all('p')
                    if paragraphs:  # Only if paragraphs found
                        for p in paragraphs:
                            text = p.get_text(strip=True)
                            if text and len(text) > 20:  # Skip very short paragraphs
                                content += text + " "
                        content_found = True
                        break
                if content_found:
                    break

        # Fallback: ambil paragraf dari body utama
        if not content:
            main_content = soup.select_one('main') or soup.select_one('article') or soup
            if main_content:
                paragraphs = main_content.find_all('p')
                for p in paragraphs[:8]:  # Limit to first 8 paragraphs
                    text = p.get_text(strip=True)
                    if text and len(text) > 20:
                        content += text + " "

        # Clean content
        content = content.replace('\n', ' ').replace('\t', ' ')
        content = ' '.join(content.split())  # Remove extra spaces

        # Remove common unwanted text for Kompas
        unwanted_phrases = [
            'ADVERTISEMENT', 'SCROLL TO RESUME CONTENT',
            'Baca juga:', 'Simak Video', 'BACA JUGA:',
            'Halaman selanjutnya', 'Lanjutkan membaca',
            'KOMPAS.com', 'Dapatkan update berita pilihan',
            'Tulis komentarmu', 'Lihat Semua', 'Editor :'
        ]

        for phrase in unwanted_phrases:
            content = content.replace(phrase, '')

        content = content.strip()

        return content[:7000]  # Limit content length

    except Exception as e:
        print_status(f"Content extraction failed for {url}: {e}", "ERROR")
        return ""

def scrape_search_results(query='uu tni', pages=3):
    global processed_urls
    all_articles = []
    processed_urls.clear()  # Reset processed URLs

    print_header(f"Scraping Kompas.com for: '{query}'")
    print(f"Pages to scrape: {pages}")

    for page in range(1, pages + 1):
        print_status(f"Processing page {page}/{pages}")
        
        # Kompas search URL format
        url = f'https://search.kompas.com/search?q={query}&page={page}'

        try:
            response = session.get(url)
            response.raise_for_status()

            soup = bs(response.text, 'lxml')

            # Debug: Print page structure info
            print_status(f"Page {page} HTML length: {len(response.text)} characters")
            
            # Check if page has content
            if len(response.text) < 1000:
                print_status(f"Page {page} seems empty or blocked", "WARNING")
                continue

            # Analyze structure on every page for debugging
            analyze_structure(url)

            # Find unique articles
            articles = find_unique_articles(soup)
            print_status(f"Found {len(articles)} valid articles")

            for i, article in enumerate(articles):
                try:
                    print_status(f"Processing article {i+1}/{len(articles)}: {article['title'][:60]}...")

                    # Extract full content
                    content = extract_article_content(article['url'])

                    if content and len(content) > 50:  # Only save articles with substantial content
                        full_article = {
                            'title': article['title'],
                            'date': article['date'],
                            'url': article['url'],
                            'content': content
                        }

                        all_articles.append(full_article)
                        print_status(f"Article saved ({len(content)} characters)", "SUCCESS")
                    else:
                        print_status("Skipped - insufficient content", "WARNING")

                except Exception as e:
                    print_status(f"Processing failed: {e}", "ERROR")
                    continue

            print_status(f"Page {page} completed. Total articles: {len(all_articles)}")
            time.sleep(2)  # Delay between pages

        except Exception as e:
            print_status(f"Page {page} failed: {e}", "ERROR")
            continue

    return all_articles

def save_to_csv(articles, filename='kompas_articles.csv'):
    try:
        # Remove any remaining duplicates based on URL
        unique_articles = []
        seen_urls = set()

        for article in articles:
            if article['url'] not in seen_urls:
                unique_articles.append(article)
                seen_urls.add(article['url'])

        with open(filename, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Title', 'Date', 'URL', 'Content'])

            for article in unique_articles:
                writer.writerow([
                    article['title'],
                    article['date'],
                    article['url'],
                    article['content']
                ])

        print_header("Scraping Results Summary")
        print(f"Unique articles saved: {len(unique_articles)}")
        print(f"Duplicate articles removed: {len(articles) - len(unique_articles)}")
        print(f"Output file: {filename}")

    except Exception as e:
        print_status(f"Failed to save CSV: {e}", "ERROR")

def print_summary(articles):
    if not articles:
        print_status("No articles found!", "WARNING")
        return

    print_header("Scraping Summary")
    print(f"Total articles collected: {len(articles)}")
    print(f"Unique URLs: {len(set(a['url'] for a in articles))}")

    print("\nSample Articles:")
    for i, article in enumerate(articles[:3]):
        print(f"\nArticle {i+1}:")
        print(f"Title: {article['title'][:80]}...")
        print(f"Date: {article['date']}")
        print(f"URL: {article['url']}")
        print(f"Content Preview: {article['content'][:100]}...")
        print(f"Content Length: {len(article['content'])} characters")

In [10]:
if __name__ == "__main__":
    print("Starting Kompas.com scraper (Fixed - No Duplicates)...")
    articles = scrape_search_results(query='uu+tni', pages=8)

    if articles:
        save_to_csv(articles)
        print_summary(articles)
    else:
        print("No articles found. Check the structure analysis output for debugging.")

Starting Kompas.com scraper (Fixed - No Duplicates)...

        SCRAPING KOMPAS.COM FOR: 'UU+TNI'         
Pages to scrape: 8
[INFO] Processing page 1/8
[INFO] Page 1 HTML length: 150472 characters

             HTML STRUCTURE ANALYSIS              
Page Title: Berita Terkini Hari Ini, Kabar Akurat Terpercaya - Kompas.com

Container Detection:
- articleList -list: 1 found
- articleList variants: 1 found
- latest__item: 0 found
- article__list--item: 0 found
- latest variants: 1 found
- article variants: 161 found
- article tags: 0 found
- item variants: 37 found
- list variants: 1 found

Best Container: div[class*="article"]
Main Article Containers Found: 161

Sample Container Analysis:
- CSS Classes: articleList, -list
- Article URL: https://nasional.kompas.com/read/2025/05/14/06300091/prospek-pembatalan-perubahan-uu-tni
- Title Preview: Prospek Pembatalan Perubahan UU TNINews14 Mei 2025...
[INFO] Using selector: div[class*="article"]
[INFO] Processing 161 article containers
[INFO] Fo

/tmp/ipykernel_35/3096124697.py:154: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  potential_titles = container.find_all(text=True)


[SUCCESS] Article saved (3093 characters)
[INFO] Processing article 2/20: Panglima TNI Sebut Penjagaan Kejaksaan Sesuai UU TNI yang Ba...
[SUCCESS] Article saved (2582 characters)
[INFO] Processing article 3/20: Mahfud MD Sebut TNI Jaga Kejaksaan Tak Sesuai UUNews16 Mei 2...
[SUCCESS] Article saved (3456 characters)
[INFO] Processing article 4/20: MK Kabulkan Penarikan Uji Materi UU TNI dari Dosen UnhanNews...
[SUCCESS] Article saved (3066 characters)
[INFO] Processing article 5/20: Mahasiswa Hukum UII Kecam Intimidasi terhadap Penggugat UU T...
[SUCCESS] Article saved (3010 characters)
[INFO] Processing article 6/20: Yusril Sebut Tugas TNI Melindungi Jaksa Tak Bertentangan den...
[SUCCESS] Article saved (2632 characters)
[INFO] Processing article 7/20: Soal TNI Amankan Kejaksaan, Puan: Sesuai Aturan di UU TNI da...
[SUCCESS] Article saved (3228 characters)
[INFO] Processing article 8/20: Mahfud Anggap Pengerahan TNI Jaga Kejaksaan Bukan karena Rev...
[SUCCESS] Article saved (2750 char